# Soil Moisture — fixing the evaluation path

This is an **updated experiment**, not a claim that I improved the original score. I discovered that the old notebook fitted `StandardScaler` before splitting the data. Here I show why splitting first matters and create a reproducible evaluation path.

## 1. Where did the original data go?

The original notebook used `t_s1_am_6am.csv`. If it is unavailable, run the synthetic demonstration cell only. **Do not treat synthetic scores as soil-model performance.**

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

DATA_FILE = Path("data/t_s1_am_6am.csv")
REQUIRED = ["VV", "VH", "smap_am", "soil_moisture"]
print("Data present?", DATA_FILE.exists())

## 2. Synthetic smoke test for the pipeline (NOT a soil evaluation)

This is merely a controlled fake dataset. Because we generated the rule ourselves, we can see whether the code finds the expected signal. This tests the machinery, not the old scientific claim.

In [ ]:
rng = np.random.default_rng(42)
X_demo = pd.DataFrame(rng.normal(size=(300, 3)), columns=["VV", "VH", "smap_am"])
y_demo = 1.3 * X_demo["VV"] - 0.4 * X_demo["VH"] + rng.normal(scale=0.15, size=len(X_demo))

X_train, X_test, y_train, y_test = train_test_split(X_demo, y_demo, test_size=0.2, random_state=42)
model = Pipeline([("imputer", SimpleImputer(strategy="median")),
                  ("scaler", StandardScaler()),
                  ("regressor", LinearRegression())])
model.fit(X_train, y_train)
print("Synthetic fixture R2 (NOT actual soil):", round(r2_score(y_test, model.predict(X_test)), 4))
print("Scaler trained on", model.named_steps["scaler"].n_samples_seen_, "TRAIN rows, not all rows")
assert model.named_steps["scaler"].n_samples_seen_ == len(X_train)

## 3. The proper real-data evaluation

Only runs when the original CSV is supplied. We do **not** filter by target/outlier thresholds before splitting. We compare against a constant-mean baseline.

In [ ]:
if not DATA_FILE.exists():
    print("Original CSV is missing — real-data metrics intentionally not computed.")
else:
    df = pd.read_csv(DATA_FILE)
    assert set(REQUIRED).issubset(df.columns), f"Expected: {REQUIRED}"
    df = df[REQUIRED].copy()
    df[REQUIRED] = df[REQUIRED].apply(pd.to_numeric, errors="coerce")
    df = df.dropna(subset=["soil_moisture"])
    X = df[["VV", "VH", "smap_am"]]
    y = df["soil_moisture"]
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
    models = {
        "Mean baseline": Pipeline([("imputer", SimpleImputer(strategy="median")),
                                   ("model", DummyRegressor(strategy="mean"))]),
        "Linear regression": Pipeline([("imputer", SimpleImputer(strategy="median")),
                                       ("scaler", StandardScaler()),
                                       ("model", LinearRegression())])
    }
    rows = []
    for name, candidate in models.items():
        candidate.fit(X_train, y_train)
        pred = candidate.predict(X_test)
        rows.append({"model":name,"MSE":mean_squared_error(y_test,pred),
                     "MAE":mean_absolute_error(y_test,pred),"R2":r2_score(y_test,pred)})
    display(pd.DataFrame(rows).round(5))

## 4. Failure mechanism

`StandardScaler.fit(X)` computes a mean and variance. If `X` includes test observations, the training process indirectly sees statistics of those observations. A safe pipeline instead fits the scaler on training rows only, then transforms held-out test rows using the *training* mean and variance.

**Question for my future self:** If the real test R² remains near zero after we fix leakage, what should I conclude about the available predictors?